# NB-02-01 — The Agent Loop in a Gridworld

Five architectures, one environment, measured. Every number comes from `code/agents.py` and
`code/environments.py` with seed 1; nothing is hardcoded.

Read alongside L02-04 and L02-05.

In [1]:
import pathlib, sys

CODE = None
for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (base / "code" / "agents.py").exists():
        CODE = base / "code"
        break
if CODE is None:
    CODE = pathlib.Path.cwd().parent / "code"
sys.path.insert(0, str(CODE))

from environments import GridWorld, ACTIONS
from agents import (RandomAgent, ReflexAgent, ModelBasedAgent,
                    GoalBasedAgent, UtilityBasedAgent, LearningAgent, run_episode)
print("actions:", ACTIONS, "| code:", CODE)

actions: ('up', 'down', 'left', 'right', 'suck', 'noop') | code: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M02-intelligent-agents-and-environments\code


## 1. The environment and its percept

A 4x4 floor with two dirty cells. `GridWorld(observable="local")` returns the *local* view: my cell,
whether it is dirty, the four wall flags, and whether I just bumped. The default `"full"` setting hands
the agent the whole state, which is a different (and much easier) problem.

In [2]:
world = GridWorld(width=4, height=4, dirt={(1, 1), (3, 2)}, observable="local", seed=1)
print("local percept:", world.percept())
print("shape: (kind, position, dirty_here, walls_nsew, bump)")
reward = world.step("right")
print("after 'right': reward", reward, "| new percept:", world.percept())

full = GridWorld(width=4, height=4, dirt={(1, 1), (3, 2)}, observable="full", seed=1)
print("full percept: ", full.percept())

local percept: ('local', (0, 0), False, (True, False, True, False), False)
shape: (kind, position, dirty_here, walls_nsew, bump)
after 'right': reward -1 | new percept: ('local', (1, 0), False, (True, False, False, False), False)
full percept:  ('full', (0, 0), frozenset({(1, 1), (3, 2)}), frozenset())


## 2. Five architectures on the same floor

60-step limit, local percept. `performance` is the environment's score: reward for cleaning, cost per step.

In [3]:
rows = []
for factory in (RandomAgent, ReflexAgent, ModelBasedAgent, GoalBasedAgent, UtilityBasedAgent):
    world = GridWorld(width=4, height=4, dirt={(1, 1), (3, 2)}, observable="local", seed=1)
    world.reset(dirt={(1, 1), (3, 2)})
    result = run_episode(factory(), world, max_steps=60)
    rows.append(result)

print(f"{'agent':15s} {'steps':>6s} {'cleaned':>8s} {'score':>7s}")
for r in rows:
    print(f"{r['agent']:15s} {r['steps']:6d} {r['cleaned']:8d} {r['performance']:7d}")

agent            steps  cleaned   score
random              60        1     -34
reflex              60        1     -49
model-based         15        2       7
goal-based          15        2       7
utility-based       15        2       7


## 3. Why the reflex agent fails

It cleans the cell it stands on, then follows a fixed cycle. With a local percept it cannot represent
"where have I been", so on a two-room floor it runs out of steps having cleaned one cell.

In [4]:
world = GridWorld(width=4, height=4, dirt={(1, 1), (3, 2), (0, 3)}, observable="local", seed=2)
reflex = run_episode(ReflexAgent(), world, max_steps=80)
world.reset(dirt={(1, 1), (3, 2), (0, 3)})
model = run_episode(ModelBasedAgent(), world, max_steps=80)
print("reflex     :", reflex)
print("model-based:", model)
print("difference in cells cleaned:", model["cleaned"] - reflex["cleaned"])

reflex     : {'agent': 'reflex', 'steps': 80, 'cleaned': 1, 'performance': -69}
model-based: {'agent': 'model-based', 'steps': 16, 'cleaned': 3, 'performance': 17}
difference in cells cleaned: 2


## 4. Local versus full percept

The same architectures under both percepts. Note which agents improve: capability is a property of the
percept as much as of the code.

In [5]:
for observable in ("local", "full"):
    print("---", observable)
    for factory in (ReflexAgent, ModelBasedAgent, GoalBasedAgent):
        w = GridWorld(width=4, height=4, dirt={(1, 1), (3, 2)}, observable=observable, seed=1)
        w.reset(dirt={(1, 1), (3, 2)})
        print("  ", run_episode(factory(), w, max_steps=60))

--- local
   {'agent': 'reflex', 'steps': 60, 'cleaned': 1, 'performance': -49}
   {'agent': 'model-based', 'steps': 15, 'cleaned': 2, 'performance': 7}
   {'agent': 'goal-based', 'steps': 15, 'cleaned': 2, 'performance': 7}
--- full
   {'agent': 'reflex', 'steps': 60, 'cleaned': 1, 'performance': -49}
   {'agent': 'model-based', 'steps': 15, 'cleaned': 2, 'performance': 7}
   {'agent': 'goal-based', 'steps': 7, 'cleaned': 2, 'performance': 15}


## 5. Learning improves with episodes

`LearningAgent` updates a Q-table after every step. First five episodes are poor; later ones are not.
It does not converge in 20 episodes, and the lesson says why.

In [6]:
world = GridWorld(width=4, height=4, dirt={(1, 1), (3, 2)}, observable="local", seed=1)
learner = LearningAgent(epsilon=0.2, seed=0)
scores = []
for episode in range(20):
    world.reset(dirt={(1, 1), (3, 2)})
    scores.append(run_episode(learner, world, max_steps=60, learn=True)["performance"])

print("scores:", scores)
print("first five mean:", round(sum(scores[:5]) / 5, 1))
print("last five mean: ", round(sum(scores[-5:]) / 5, 1))
print("states in table:", len(learner.q))

scores: [-28, -24, -10, -13, -8, -3, -7, -29, 9, -20, -2, -5, -5, -22, -15, -8, -12, -10, 11, 15]
first five mean: -16.6
last five mean:  -0.8
states in table: 18


## 6. Checks

Determinism, belief consistency, and the claim that the reflex step count never improves with a longer
budget — the loop, not the limit, is the problem.

In [7]:
def run(factory, dirt, steps=60):
    w = GridWorld(width=4, height=4, dirt=set(dirt), observable="local", seed=1)
    w.reset(dirt=set(dirt))
    return run_episode(factory(), w, max_steps=steps)

a = run(ReflexAgent, {(1, 1), (3, 2)})
b = run(ReflexAgent, {(1, 1), (3, 2)})
print("PASS  deterministic" if a == b else "FAIL  deterministic", a)

long_budget = run(ReflexAgent, {(1, 1), (3, 2)}, steps=200)
print("reflex cleaned at 60 steps:", a["cleaned"], "| at 200 steps:", long_budget["cleaned"])

def run_full(factory, dirt, steps=60):
    w = GridWorld(width=4, height=4, dirt=set(dirt), observable="full", seed=1)
    w.reset(dirt=set(dirt))
    return run_episode(factory(), w, max_steps=steps)


local_goal = run(GoalBasedAgent, {(1, 1), (3, 2)})
full_goal = run_full(GoalBasedAgent, {(1, 1), (3, 2)})
full_model = run_full(ModelBasedAgent, {(1, 1), (3, 2)})
print("goal-based, local percept:", local_goal["steps"], "steps | full percept:", full_goal["steps"], "steps")
print("PASS  planning pays only when the goal is observable"
      if full_goal["steps"] < full_model["steps"] and local_goal["steps"] == run(ModelBasedAgent, {(1, 1), (3, 2)})["steps"]
      else "FAIL  planning comparison")

agent = ModelBasedAgent()
w = GridWorld(width=4, height=4, dirt={(1, 1)}, observable="local", seed=1)
agent.program(w.percept())
print("PASS  belief updated" if agent.visited else "FAIL  belief updated", sorted(agent.visited))

PASS  deterministic {'agent': 'reflex', 'steps': 60, 'cleaned': 1, 'performance': -49}
reflex cleaned at 60 steps: 1 | at 200 steps: 1
goal-based, local percept: 15 steps | full percept: 7 steps
PASS  planning pays only when the goal is observable
PASS  belief updated [(0, 0)]


## What you should take away

- With a local percept the same floor separates the architectures: 60 steps and one cell (reflex)
  against 15 steps and two cells (model-based and goal-based alike).
- Planning only pays when the goal is visible: with a full percept the goal-based agent reaches the
  same result in 7 steps.
- A local percept is what makes the reflex agent's rules incomplete; adding memory, not tuning rules,
  is the fix.
- Utility and goal agents tie here, and the utility agent wins only when costs differ or outcomes are
  uncertain.
- Learning shows a trend within 20 episodes; convergence needs far more experience and a decaying rate.

Next: [LAB-02-01](../labs/lab-02-01-intelligent-vacuum-cleaner.md) and
[LAB-02-02](../labs/lab-02-02-campus-navigation-agent.md).